# OLAF : creating a simple pipeline demo

In this demo, we create a simple pipeline using components from the OLAF library. The corpus is composed of basic sentences. We want to extract concepts and relations from it.

In [1]:
import spacy

In [2]:
import pickle
import dill
import os



# Import all necessary items from the olaf package
from olaf import Pipeline
from olaf.pipeline.pipeline_component.term_extraction import (
    POSTermExtraction,
    TFIDFTermExtraction,
    ManualCandidateTermExtraction
    )
from olaf.pipeline.pipeline_component.concept_relation_extraction import (
    CTsToConceptExtraction,
    CTsToRelationExtraction,
    SynonymRelationExtraction,
    SynonymConceptExtraction,
    AgglomerativeClusteringRelationExtraction,
    AgglomerativeClusteringConceptExtraction,
    LLMBasedConceptExtraction,
    LLMBasedRelationExtraction
)
from olaf.pipeline.pipeline_component.axiom_extraction.owl_axiom_extraction import OWLAxiomExtraction
from olaf.data_container.knowledge_representation_schema import KnowledgeRepresentation
from olaf.repository.serialiser import BaseOWLSerialiser
from olaf.repository.corpus_loader.text_corpus_loader import TextCorpusLoader
from olaf.data_container import CandidateTerm, Relation, Concept


/home/talibe/Bureau/Stage Insa/OLAF Research/olaf/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [36]:
#ajouter par NFD
!python -m spacy download fr_core_news_lg

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


     ━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━━━━━━━━━ 250.5/571.8 MB 17.5 MB/s eta 0:00:19
ERROR: Operation cancelled by user
^C


In [3]:
# Load the spacy language model according to the corpus
spacy_model = spacy.load("fr_core_news_lg")

In [4]:
# Initialise the corpus (for this example text version)
corpus = [

"""La norme NF G 00-001 détaille les défauts que nous pouvons retrouvés lors d'une inspection par Vision Industrielle par exemple. Dans cette partie, nous aborderons les défauts que nous pourrions rencontrer durant nos inspections sur des textiles en défilement.

Tout d'abord, une pièce de tissu peut vulgairement être nommée « morceau de tissu ». En effet, il s'agit d'un échantillon venant d'un ensemble. Les principaux défauts que nous trouvons sont les suivants :

Variation de nuance : La nuance est « non uniforme » ;

Nuance non conforme : La nuance n'est pas celle qui était attendue;

Irrégularité de motif : Le motif n'a pas une répétition régulière;

Défaut d'embuvage : Excès d'embuvage ou insuffisance d'embuvage;

Répartition asymétrique du motif : Motif non identique sur les deux moitiés;

Marquage défectueux : Non conforme aux usages demandés;

Sinuosité dans le sens de la longueur : Pièce non rectiligne.

 Lors de la teinture ou de l'impression, il y a des défauts qui peuvent apparaitre, les principaux sont les suivants :

Barre de finition;

Rayure de finition;

Cassure : Faux pli;

Tache de teinture : Nuance localement altérée;

Défaut d'impression : Impression irrégulière : trait de racle, manque de couleur, ...;

Marque de couture : Couture apparente alors que ce n'est pas prévu;

Trou de picots : Déchirure provoquée par une prise du tissu mauvaise.

La lisière est ce qui termine les côtés d'un tissu. Autrement dit, il s'agit de la partie où la trame s'est bouclée par le retour de la navette sur elle-même. La navette est l'outil permettant de réaliser la trame. Il arrive souvent que la lisière est dans un autre tissu ou bien d'une autre couleur que la base. Cela peut être un avantage lors des contrôles, car nous pouvons ainsi différencier facilement les éléments, mais cela peut être un inconvénient car il ne faut pas détecter ceci comme un défaut. Les principaux défauts que nous pouvons trouver sur la lisière sont les suivants :

Lisière rompue : Lisière ayant une coupure ou déchirure.

Lisière ondulée (lisière ballante) : Tension de la lisière insuffisante pouvant faire onduler le tissu.

Lisière roulée : La lisière est repliée sur elle-même.

Lisière surtendue (lisière tirante) : Tension lisière supérieure à celle du tissu.

Lisière crénelée : Le bord de la lisière est irrégulier, présence de petits créneaux.

Les fils sont une source de défauts relativement importante. Un soin tout particulier doit y être consacré. Nous pouvons retrouver ceux-ci :

Irrégularités localisées des fils : Les fils ne sont pas réguliers.

Défaut de torsion : Torsion non adaptée par rapport au fil.

Fil fin : Portion de fil de masse linéique inférieure à la masse linéique normalement utilisée.

Gros fil : Portion de fil de masse linéique supérieure à la masse linéique normalement utilisée.

Flamme ou grosseur : Sur-épaisseur plus ou moins importante.

Vrille : Boucle torsadée de petite taille, faite par un fil.

Travelage : Fil qui s'enroule entre des éléments plus petits.

Fils étrangers : Fils différents de ceux présents.

Fils souillés : Coloration pas normale de fils.

La trame est constituée des fils qui traversent/croisent ceux de la chaîne suivant l'armure que nous souhaitons avoir, en un mouvement de va et vient. Sur cette photo venant de « L'encyclopédie Larousse 2002 », les fils de trame sont ceux en vert alors que ceux de la chaîne sont ceux en bleu et en rouge. Une duite est l'ensemble du tissage entre le fil/les fils de trame et le fil/les fils de chaîne. Les principaux défauts sont les suivants :

Double duite : Deux duites sont présentes.

Eboulure : Petite partie de duite repliée sur elle-même.

Duite rompue : Duite présentant une coupure ou une cassure.

Duite détendue : Duite dont la longueur est supérieure à celle des duites voisines.

Duite manquante : Absence d'une duite.

Duite manquée (duite sautée, pas failli) : Duite incorrectement insérée

Duite tendue : Duite dont la longueur est inférieure aux autres duites

Duite tirante : Partie de duite sans embuvage pouvant provoquer un froncement de tissu.

Barre : Existence d'une bande plus ou moins étroite se distinguant du reste du tissu.

Irrégularité du duitage : Barre claire due à un duitage insuffisant ou barre serrée due à un duitage trop important par rapport au duitage du tissu.

La chaîne représente l'ossature de la pièce. Ce sont les fils qui sont dans la longueur de la pièce. Dans la partie précédente 2.2.5 c'est les fils rouges et les fils bleus. Les défauts de chaîne sont les suivants :

Fil de chaîne double : Deux fils de chaîne sont involontairement tissés ensemble.

Fil de chaîne rompu : Fil de chaîne ayant une coupure ou une cassure.

Fil de chaîne flottant : Fil de chaîne pas assez tendu.

Fil de chaîne manquant (fil couru) : Absence d'un fil de chaîne.

Défaut de peigne (roselage) : Espace anormal entre des fils de chaîne.

La liste des défauts ou caractéristiques pouvant être observée avec un système de Vision Industrielle est loin d'être fixe. Voici quelques autres défauts/caractéristiques pouvant être présentes :

Amas de fibres : Forte concentration sur le tissu de fibres prises dans les filé ou dans le tissu.

Duvet : Amas de fibres courtes. On parle de "petit duvet " lorsqu'il s'agit de duvet de longueur inférieure à 4 mm et de gros duvet quand c'est supérieure à 10 mm.

Volard : Retombée de déchets

Bouton (neps) : Accumulation accidentelle de fibres comportant un noyau prononcé.

Corps étranger : Matière étrangère, autre que la fibre textile, ou matière de même nature mais ayant un comportement différent, se trouvant accidentellement incorporée dans le tissu.

Tache de variation de compte en chaîne ou en trame : Déformation locale sensiblement ronde due à un glissement, des fils de chaîne, par frottement.

Goutte de pluie : Augmentation ou diminution, sur une très courte longueur, de l'amplitude des ondulations de plusieurs fils.

Coupure : Incision dans le tissu.

Déchirure : Ouverture dans le tissu provoquée par la rupture de plusieurs fils de chaîne.

Trou : Ouverture dans le tissu, provoquée par la rupture ou la destruction d'un ou plusieurs fils de chaîne.

Clairière (crapaud, nid, pas de chat, patte de poule) : Déformation locale d'un tissu due au déplacement d'un ou plusieurs fils de même sens, avec éventuellement rupture d'un ou plusieurs fils de l'autre sens.

Accroc : Double déchirure du tissu provoquée généralement par une pointe.

Sauté (piqûre, bride) : Présence d'un fil de chaîne ou d'une duite qui n'est pas lié au fil auquel il devrait l'être.

Noeud pris : Noeud du fil de chaîne/duite retenant une duite ou un fil de chaîne sur une courte distance et provoquant un trou de forme triangulaire sur le tissu.

Point : Sur-épaisseur ponctuelle provoquée par un nœud.

Ebouriffage local : Présence de fils endommagés ou rompus donnant à la surface du tissu, une apparence poilue.

Frappe de navette (portée de navette) : Marque due à la détérioration des fils de chaîne par la navette.

Ondulation (cloquage, gondolage) : Déformation accidentelle empêchant le tissu de reposer à plat sur une surface horizontale."""
]

In [5]:
bad_concept_pos = ["VERB","ADV","ADP","CCONJ","DET", "INTJ", "NUM","PRON", "PART", "SCONJ"]
bad_relation_pos = ["NOUN","ADV","ADP","CCONJ","DET", "INTJ", "NUM","PRON", "PART", "SCONJ"]
def get_bad_pos (bad_pos):
    def candidates_post_processing(candidates: set[CandidateTerm]) -> set[CandidateTerm]:
        
        list_label = []
        new_candidates = set()
        for candidate in candidates:
            keep = True
            if len(candidate.corpus_occurrences) > 0:
                for co in candidate.corpus_occurrences:
                    for token in co:
                        if (token.is_punct or token.is_stop or token.pos in bad_pos):
                            keep = False
                            break
            else:
                keep = False
            if keep and candidate.label not in list_label:
                new_candidates.add(candidate)
                list_label.append(candidate.label)
        return new_candidates
    return candidates_post_processing

In [6]:
# relation candidates extraction

relation_pos = ["VERB"]

tfidf_relation_term_extraction = TFIDFTermExtraction( # documentation à regarder
    max_term_token_length=3,
    candidate_term_threshold=0.04,
    cts_post_processing_functions=[get_bad_pos(bad_relation_pos)]
)
pos_relation_term_extraction = POSTermExtraction(pos_selection=relation_pos)
# pos_relation_term_extraction = ManualCandidateTermExtraction(

# )

# relation  extraction


ct_relation_extraction = CTsToRelationExtraction(concept_max_distance=6)
synonym_relation_extraction =  SynonymRelationExtraction(concept_max_distance=6)
agglo_relation_extraction = AgglomerativeClusteringRelationExtraction(distance_threshold=.4)
# llm_relation_extraction = LLMBasedRelationExtraction() # ne pas utiliser pour le moment


[2025-05-13 00:59:02,599] [WARNING] [tfidf_term_extraction] [_check_parameters] [Selected token sequence document attribute not set by the user.
                By default the system will use the entire content of the document.]
[2025-05-13 00:59:02,600] [WARNING] [pos_term_extraction] [__init__] [No preprocessing function provided for spans. Using the default one.]
[2025-05-13 00:59:02,600] [WARNING] [pos_term_extraction] [_check_parameters] [POS term extraction token sequence attribute not set by the user.
               By default the system will use the entire content of the document.]
[2025-05-13 00:59:02,601] [WARNING] [agglomerative_clustering_relation_extraction] [_check_parameters] [No value given for embedding_model parameter, default will be set to all-mpnet-base-v2.]
[2025-05-13 00:59:02,601] [WARNING] [agglomerative_clustering_relation_extraction] [_check_parameters] [No value given for metric option, default will be set to cosine.]
[2025-05-13 00:59:02,601] [WARNING] [agg

In [7]:
# concept candidates extraction

concepts_pos = ["NOUN"]

tfidf_concept_term_extraction = TFIDFTermExtraction( # documentation à regarder
    max_term_token_length=2,
    candidate_term_threshold=0.04,
    cts_post_processing_functions= [get_bad_pos(bad_concept_pos)]
)
pos_concept_term_extraction = POSTermExtraction(pos_selection=concepts_pos)
# pos_concept_term_extraction = ManualCandidateTermExtraction()

# concept  extraction


ct_concept_extraction = CTsToConceptExtraction()
synonym_concept_extraction = SynonymConceptExtraction()
agglo_concept_extraction = AgglomerativeClusteringConceptExtraction(distance_threshold=.4)
# llm_concept_extraction = LLMBasedConceptExtraction() # ne pas utiliser pour le moment

[2025-05-13 00:59:02,606] [WARNING] [tfidf_term_extraction] [_check_parameters] [Selected token sequence document attribute not set by the user.
                By default the system will use the entire content of the document.]
[2025-05-13 00:59:02,606] [WARNING] [pos_term_extraction] [__init__] [No preprocessing function provided for spans. Using the default one.]
[2025-05-13 00:59:02,607] [WARNING] [pos_term_extraction] [_check_parameters] [POS term extraction token sequence attribute not set by the user.
               By default the system will use the entire content of the document.]
[2025-05-13 00:59:02,607] [WARNING] [agglomerative_clustering_concept_extraction] [_check_parameters] [No value given for embedding_model parameter, default will be set to all-mpnet-base-v2.]
[2025-05-13 00:59:02,607] [WARNING] [agglomerative_clustering_concept_extraction] [_check_parameters] [No value given for metric option, default will be set to cosine.]


In [8]:
def clean_relations(kr: KnowledgeRepresentation):
    """
    Clean the relations in the knowledge representation
    :param kr: KnowledgeRepresentation
    :return: None
    """
    relations_to_remove = []
    for relation in kr.relations:
        if relation.source_concept is None or relation.destination_concept is None:
            relations_to_remove.append(relation)
        elif relation.source_concept.label == relation.destination_concept.label:
            relations_to_remove.append(relation)
    for relation in relations_to_remove:
        kr.relations.remove(relation)

def serialize_pipeline(pipeline, file_path):
    """
    Serialize the pipeline to a file
    :param pipeline: Pipeline
    :param file_path: str
    :return: None
    """
    components = pipeline.pipeline_components
    with open(file_path, 'wb') as f:
        dill.dump(components, f)

def deserialize_pipeline(file_path):
    """
    Deserialize the pipeline from a file
    :param file_path: str
    :return: Pipeline
    """
    with open(file_path, 'rb') as f:
        components = dill.load(f)
    return Pipeline(
        spacy_model=spacy_model,
        pipeline_components=components,
    )


# Pipeline 1
     - POSTermExtraction
     - CTsToConceptExtraction
     - POSTermExtraction
     - CTsToRelationExtraction


In [ ]:
pipeline_1 = Pipeline(
    spacy_model=spacy_model,
    pipeline_components=[
       	pos_concept_term_extraction, 
        CTsToConceptExtraction(),
        pos_relation_term_extraction, 
        CTsToRelationExtraction()],
    corpus=[doc for doc in spacy_model.pipe(corpus)]
)

pipeline_1.run()

# Clean the relations
clean_relations(pipeline_1.kr)



[2025-05-13 01:05:45,732] [WARNING] [candidate_terms_to_relations] [_check_parameters] [No value given for concept_max_distance parameter, default will be set to 5.]


In [27]:
for concept in pipeline_1.kr.concepts:
  print(concept)

gondolage
répartition
duitage
-
poule
avantage
sens
nid
masse
compte
filé
tissu
tache
chaîne
distance
défaut
cassure
sauté
déchets
pièce
racle
duite
retombée
mm
inconvénient
crapaud
manque
créneaux
piqûre
système
apparence
reste
vrille
sinuosité
fils
armure
usages
destruction
couture
défilement
rouge
effet
taille
vert
vision
nature
nœud
mouvement
finition
accumulation
outil
norme
prise
froncement
inspection
ossature
noyau
amas
rayure
00
motif
trame
diminution
tension
comportement
contrôles
déformation
morceau
pointe
concentration
éléments
nf
source
plat
irrégularité
ci
peigne
portion
lisière
bord
fibre
fil
nuance
moitiés
répétition
défauts
être
frottement
flamme
noeud
coupure
chat
glissement
ouverture
navette
matière
ondulations
fibres
ebouriffage
couleur
rectiligne
présence
détérioration
coloration
frappe
ondulation
001
grosseur
rapport
trou
absence
caractéristiques
amplitude
variation
textiles
photo
cloquage
pli
incision
fin
ensemble
impression
encyclopédie
embuvage
inspections
excès

In [28]:
for relation in pipeline_1.kr.relations:
  if relation.source_concept is not None or relation.destination_concept is not None:
    print(relation.source_concept, "-",  relation, "-", relation.destination_concept)


nf - détaille - défauts
tissu - roulée - lisière
001 - détaille - défauts
ouverture - provoquée - rupture
vrille - torsadée - taille
usages - demandés - sinuosité
tissu - reposer - plat
être - retrouver - -
déchirure - provoquée - tissu
noeud - pris - fil
outil - réaliser - trame
tissu - provoquée - rupture
caractéristiques - observée - vision
distance - provoquant - trou
lisière - ayant - déchirure
tissu - prises - filé
torsion - adaptée - rapport
être - pouvons - -
chaîne - retenant - fil
fibres - prises - filé
comportement - trouvant - tissu
tissu - représente - pièce
caractéristiques - pouvant - être
duite - rompue - coupure
navette - portée - marque
éléments - peut - inconvénient
masse - utilisée - grosseur
duite - lié - fil
défauts - pouvons - lisière
être - présentes - fibres
teinture - altérée - défaut
tissu - reposer - surface
être - voici - caractéristiques
caractéristiques - pouvant - système
chaîne - ayant - cassure
être - pouvons - ci
contrôles - pouvons - éléments
fil - a

In [29]:
my_olaf_demo1_serialiser = BaseOWLSerialiser("http://olaf_demo_results.org/")
my_olaf_demo1_serialiser.build_graph(pipeline_1.kr)

# Export the RDF graph file path and in default format (owl)
my_olaf_demo1_serialiser.export_graph("../data/textile/textile_default_kr_1.owl")


# Serialize the pipeline
serialize_pipeline(pipeline_1, "../data/textile/textile_pipeline_fr_1.dill")

# Pipeline 2
	- TFIDFTermExtraction
	- CTsToConceptExtraction
	- TFIDFTermExtraction
	- CTsToRelationExtraction

In [30]:
pipeline_2 = Pipeline(
    spacy_model=spacy_model,
    pipeline_components=[
        TFIDFTermExtraction(
            max_term_token_length=2,
            candidate_term_threshold=0.02,
            cts_post_processing_functions=[get_bad_pos(bad_concept_pos)]
        ),
        ct_concept_extraction,
        TFIDFTermExtraction(
            max_term_token_length=2,
            candidate_term_threshold=0.01,
            cts_post_processing_functions=[get_bad_pos(bad_relation_pos)]
        ),
        ct_relation_extraction],
    corpus=[doc for doc in spacy_model.pipe(corpus)]
)

pipeline_2.run()
clean_relations(pipeline_2.kr)
print(("Concepts length: ", len(pipeline_2.kr.concepts)))

[2025-05-13 01:05:46,527] [WARNING] [tfidf_term_extraction] [_check_parameters] [Selected token sequence document attribute not set by the user.
                By default the system will use the entire content of the document.]
[2025-05-13 01:05:46,528] [WARNING] [tfidf_term_extraction] [_check_parameters] [Selected token sequence document attribute not set by the user.
                By default the system will use the entire content of the document.]
/home/talibe/Bureau/Stage Insa/OLAF Research/olaf/env/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
[2025-05-13 01:05:46,632] [WARNING] [tfidf_term_extraction] [_get_corpus_occurrences] [No corpus occurrence found for candidate term . lisière]
[2025-05-13 01:05:46,633] [WARNING] [tfidf_term_extraction] [_get_corpus_occurrences] [No corpus occurrence found for candidate term . fil]
[2025-05-13 01:05:46,633] [

('Concepts length: ', 32)


In [31]:
for concept in pipeline_2.kr.concepts:
  print(concept)

duite
défaut
masse
nuance
linéique
barre
duvet
due
pouvons
embuvage
déchirure
pièce
navette
provoquée
partie
défauts
masse linéique
non
principaux
duitage
fil
pouvant
coupure
lisière
trame
tissu
longueur
fibres
supérieure
chaîne
fils
motif


In [32]:
for relation in pipeline_2.kr.relations:
  if relation.source_concept is not None or relation.destination_concept is not None:
    print(relation.source_concept, "-",  relation, "-", relation.destination_concept)


due - barre - duitage
chaîne - fil - coupure
non - conforme - nuance
supérieure - duites - duite
masse - linéique normalement - fil
tissu - duitage - barre
fil - trame - fils
masse - gros - fil
navette - due - fils
déchirure - tissu - provoquée
fil - portion - linéique
chaîne - chaîne - fil
longueur - duite - duite
linéique - supérieure - masse
tissu - duvet - fibres
linéique - fil - masse
fil - fil - masse linéique
trame - locale - due
tissu - amas - fibres
linéique - linéique normalement - fil
déchirure - tissu - lisière
tissu - rupture - fils
longueur - duites - duite
duitage - duitage - chaîne
partie - duite - embuvage
linéique - inférieure - masse
pouvant - lisière - lisière
coupure - lisière - lisière
chaîne - principaux - défauts
linéique - inférieure - masse linéique
trame - fil - chaîne
duitage - due - barre
masse - normalement - fil
défauts - caractéristiques - pouvant
fil - supérieure - masse
embuvage - tissu - barre
longueur - duites - partie
lisière - rompue - coupure
duve

In [33]:
my_olaf_demo2_serialiser = BaseOWLSerialiser("http://olaf_demo_results.org/")
my_olaf_demo2_serialiser.build_graph(pipeline_2.kr)

# Export the RDF graph file path and in default format (owl)
my_olaf_demo2_serialiser.export_graph("../data/textile/textile_default_kr_2.owl")


# Serialize the pipeline
serialize_pipeline(pipeline_2, "../data/textile/textile_pipeline_fr_2.dill")

# Pipeline 3
	- TFIDFTermExtraction
	- AgglomerativeClusteringConceptExtraction
	- TFIDFTermExtraction
	- AgglomerativeClusteringRelationtExtraction

In [34]:
pipeline_3 = Pipeline(
    spacy_model=spacy_model,
    pipeline_components=[
        tfidf_concept_term_extraction,
        agglo_concept_extraction,
        tfidf_relation_term_extraction,
        agglo_relation_extraction],
    corpus=[doc for doc in spacy_model.pipe(corpus)]
)

pipeline_3.run()
clean_relations(pipeline_3.kr)

/home/talibe/Bureau/Stage Insa/OLAF Research/olaf/env/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/home/talibe/Bureau/Stage Insa/OLAF Research/olaf/env/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [35]:
for concept in pipeline_3.kr.concepts:
  print(concept)

fils
duite
tissu
lisière
défauts
chaîne


In [36]:
for relation in pipeline_3.kr.relations:
  if relation.source_concept is not None or relation.destination_concept is not None:
    print(relation.source_concept, "-",  relation, "-", relation.destination_concept)


duite - fil - chaîne
fils - fils - défauts
fils - duite - duite
fils - chaîne - défauts
fils - chaîne - chaîne
duite - duite - chaîne
fils - chaîne - duite
chaîne - duite - fils
tissu - lisière - lisière
chaîne - duite - duite
fils - fils - chaîne
fils - défauts - chaîne
chaîne - fil - fils
lisière - lisière - tissu
duite - duite - fils


In [37]:
my_olaf_demo3_serialiser = BaseOWLSerialiser("http://olaf_demo_results.org/")
my_olaf_demo3_serialiser.build_graph(pipeline_3.kr)

# Export the RDF graph file path and in default format (owl)
my_olaf_demo3_serialiser.export_graph("../data/textile/textile_default_kr_3.owl")


# Serialize the pipeline
try:
	serialize_pipeline(pipeline_3, "../data/textile/textile_pipeline_fr_3.dill")
except Exception as e:
	print(f"Error serializing pipeline: {e}")

Error serializing pipeline: [E112] Pickling a span is not supported, because spans are only views of the parent Doc and can't exist on their own. A pickled span would always have to include its Doc and Vocab, which has practically no advantage over pickling the parent Doc directly. So instead of pickling the span, pickle the Doc it belongs to or use Span.as_doc to convert the span to a standalone Doc object.


# Pipeline 4
	- POSTermExtraction
	- AgglomerativeClusteringConceptExtraction
	- POSTermExtraction
	- AgglomerativeClusteringRelationtExtraction

In [38]:
pipeline_4 = Pipeline(
    spacy_model=spacy_model,
    pipeline_components=[
        pos_concept_term_extraction,
        agglo_concept_extraction,
        pos_relation_term_extraction,
        agglo_relation_extraction],
    corpus=[doc for doc in spacy_model.pipe(corpus)]
)

pipeline_4.run()
clean_relations(pipeline_4.kr)

In [39]:


for concept in pipeline_4.kr.concepts:
  print(concept)

ondulations
vision
nuance
patte
tension
nf
irrégularité
soin
barre
cassure
couleur
teinture
masse
fibre
finition
grosseur
détérioration
créneaux
moitiés
frottement
déplacement
photo
défilement
inconvénient
pointe
défaut
comportement
-
bande
sens
bleu
nature
contrôles
trou
caractéristiques
nid
sauté
ossature
tache
compte
sur-épaisseur
encyclopédie
amplitude
surface
échantillon
diminution
ensemble
avantage
mouvement
effet
portion
pluie
tissage
ebouriffage
duvet
impression
mm
destruction
reste
apparence
torsion
eboulure
lisière
forte
001
augmentation
navette
liste
vrille
système
source
poule
outil
excès
embuvage
rectiligne
glissement
bride
accumulation
retombée
variation
noeud
amas
peigne
nœud
gondolage
coloration
frappe
rapport
chat
ouverture
matière
insuffisance
morceau
éléments
sinuosité
g
filé
rayure
base
absence
distance
corps
être
boucle
concentration
répartition
retour
déformation
noyau
existence
manque
taille
trait
déchets
picots
armure
tissu
rupture
coupure
partie
prise
rouge
usa

In [40]:
for relation in pipeline_4.kr.relations:
  if relation.source_concept is not None or relation.destination_concept is not None:
    print(relation.source_concept, "-",  relation, "-", relation.destination_concept)


armure - avoir - mouvement
lisière - termine - côtés
lisière - termine - tissu
tissu - reposer - surface
absence - rompus - surface
être - présentes - fibre
tissu - reposer - pluie
être - voici - défaut
être - pouvons - ci
trame - constituée - fil
défaut - pouvant - système
fibre - parle - duvet
être - voici - caractéristiques
comportement - trouvant - tissu
teinture - altérée - défaut
outil - permettant - trame
caractéristiques - pouvant - amas
navette - permettant - trame
fil - ayant - cassure
être - pouvons - -
duvet - agit - longueur
fil - ayant - coupure
bande - ayant - cassure
bande - ayant - coupure
matière - ayant - comportement
effet - agit - échantillon
nature - ayant - comportement
défaut - pouvons - lisière
lisière - ayant - déchets
fibre - prises - filé
lisière - ayant - coupure
tissu - prises - filé
tissu - peut - être
noeud - pris - fil
trame - arrive - lisière
être - pris - noeud
liste - pouvant - système
être - pris - fil
fil - sont - longueur
distance - provoquant - l

In [41]:
my_olaf_demo4_serialiser = BaseOWLSerialiser("http://olaf_demo_results.org/")
my_olaf_demo4_serialiser.build_graph(pipeline_4.kr)

# Export the RDF graph file path and in default format (owl)
my_olaf_demo4_serialiser.export_graph("../data/textile/textile_default_kr_4.owl")


# Serialize the pipeline
try:
	serialize_pipeline(pipeline_4, "../data/textile/textile_pipeline_fr_4.dill")
except Exception as e:
	print(f"Error serializing pipeline: {e}")

Error serializing pipeline: [E112] Pickling a span is not supported, because spans are only views of the parent Doc and can't exist on their own. A pickled span would always have to include its Doc and Vocab, which has practically no advantage over pickling the parent Doc directly. So instead of pickling the span, pickle the Doc it belongs to or use Span.as_doc to convert the span to a standalone Doc object.
